# PySpark 14: production PySpark

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-14-production-pyspark.ipynb)

The pipeline works in a notebook. Now it has to run every night without you: from a scheduler, with parameters,
tested in CI, logging what it did, failing loudly when the input breaks its contract, and deployable to a cluster
with its Python dependencies.

This lesson turns notebook code into a production job: a project layout that separates logic from I/O, an entry
point with parameters, unit tests with a shared Spark session, logging and run metrics, schema checks at the
boundaries, packaging and `spark-submit`, deploy modes and cluster managers, orchestration, and the common
pipeline designs you will be asked to choose between.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 10: batch pipelines](pyspark-10-batch-pipelines.ipynb) and [PySpark 12: internals and monitoring](pyspark-12-internals-and-monitoring.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import logging

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Lay out a PySpark project

Keep **logic** (functions that take and return DataFrames) apart from **I/O** (reading sources, writing tables) and
from the **entry point** (parsing arguments, building the session). Logic is then testable with small in-memory
DataFrames, and the same functions run in a notebook, a test and a scheduled job.

```text
orders_pipeline/
├── pyproject.toml              # package metadata and dependencies
├── src/orders_pipeline/
│   ├── transforms.py           # pure DataFrame -> DataFrame functions
│   ├── io.py                   # read_bronze(), write_silver(): paths, formats, modes
│   ├── quality.py              # rules and the quality gate
│   └── jobs/daily_orders.py    # main(argv): parse args, build session, call the above
├── tests/
│   ├── conftest.py             # one SparkSession for the whole test run
│   └── test_transforms.py
└── conf/prod.yaml              # environment settings: paths, table names
```

**Where it matters**
- **Any job that runs on a schedule** or is shared by a team.
- **Code review:** small, named transformations are easy to read and test.

**Watch out for**
- Logic that reads tables itself cannot be tested without those tables.
- Notebook-only pipelines tend to accumulate hidden state from cells run out of order.

### Example

A transformation module in miniature: pure functions with no reads or writes, composed into one pipeline step.

In [ ]:
def standardize(orders_df):
    """Normalize status text and derive the business date."""
    return orders_df.withColumn("status", F.lower(F.trim("status"))).withColumn("order_date", F.to_date("order_ts"))


def daily_revenue(orders_df, statuses=("paid", "shipped")):
    """Revenue and order count per business date for the given statuses."""
    return (
        orders_df.where(F.col("status").isin(*statuses))
        .groupBy("order_date")
        .agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))
    )

daily_revenue(standardize(orders)).orderBy("order_date").show()

**What to notice:** Neither function knows where `orders` came from or where the result goes. In production, `io.py` passes in a
table read from bronze and writes the output; in a test, a three-row DataFrame goes in and the output is compared
with an expected one.

## Write an entry point with parameters

A job's entry point parses its arguments (business date, environment, input and output locations), builds or
gets the session, calls the logic, and writes the result. Pass everything that changes between runs as a
parameter: never derive the business date from the clock, never hard-code paths, and read credentials from the
platform's secret store.

**Where it matters**
- **Schedulers** pass the date and environment; **backfills** reuse the same job with other dates.
- **Local runs** call `main([...])` directly, as below.

**Watch out for**
- `date.today()` inside a job makes reruns and backfills write the wrong partition.
- Secrets in arguments end up in scheduler logs; use a secret store.

### Example

A `main(argv)` that publishes revenue for one business date, called here the way a scheduler would call it.

In [ ]:
import argparse


def main(argv):
    parser = argparse.ArgumentParser(description="Publish daily revenue for one business date.")
    parser.add_argument("--business-date", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args(argv)

    source = standardize(orders)  # a real job reads its input table here
    result = daily_revenue(source.where(F.col("order_date") == F.lit(args.business_date).cast("date")))
    target = os.path.join(args.output, f"order_date={args.business_date}")
    result.drop("order_date").write.mode("overwrite").parquet(target)
    return target

written = main(["--business-date", "2026-01-12", "--output", os.path.join(WORK_DIR, "gold_daily_revenue")])
spark.read.parquet(written).show()

**What to notice:** On 12 January, orders 9 (shipped, 1,180.00) and 10 (paid, 74.00) count: 2 orders, 1,254.00. The output path is
built from the parameter, so rerunning the job for the same date overwrites the same partition, and a backfill is
the same call with another date.

## Test transformations with pytest

Create one local `SparkSession` for the whole test run with a session-scoped fixture, build tiny DataFrames with
explicit schemas, and compare results without depending on row order. Cover the cases that break pipelines:
nulls, duplicate keys, empty input, boundary timestamps and time zones.

```python
# tests/conftest.py
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    session = (SparkSession.builder.master("local[1]")
               .config("spark.sql.shuffle.partitions", "1")
               .config("spark.sql.session.timeZone", "UTC").getOrCreate())
    yield session
    session.stop()

# tests/test_transforms.py
def test_daily_revenue_ignores_placed_orders(spark):
    ...
```

PySpark ships `pyspark.testing.assertDataFrameEqual`, which compares rows and schemas and prints a diff. On Spark
4.0 it needs pandas older than 3.0, so the example uses a three-line helper instead.

**Where it matters**
- **CI on every pull request,** in seconds, with no cluster.
- **Refactors:** tests pin behaviour while the implementation changes.

**Watch out for**
- Comparing `collect()` lists without sorting makes tests flaky.
- A fresh `SparkSession` per test makes the suite take minutes; share one.

### Example

Two tests in the shape pytest would run: one for a normal case, one for empty input.

In [ ]:
def assert_frames_equal(actual, expected):
    """Same schema and the same rows, in any order."""
    assert actual.schema.simpleString() == expected.schema.simpleString(), (actual.schema, expected.schema)
    assert sorted(actual.collect(), key=str) == sorted(expected.collect(), key=str)


def test_daily_revenue_ignores_placed_orders():
    source = spark.createDataFrame(
        [(1, "paid", date(2026, 1, 5), D("10.00")), (2, "placed", date(2026, 1, 5), D("99.00")),
         (3, "shipped", date(2026, 1, 6), D("5.00"))],
        "order_id int, status string, order_date date, amount decimal(12,2)",
    )
    expected = spark.createDataFrame(
        [(date(2026, 1, 5), 1, D("10.00")), (date(2026, 1, 6), 1, D("5.00"))],
        "order_date date, orders bigint, revenue decimal(22,2)",
    )
    assert_frames_equal(daily_revenue(source), expected)


def test_daily_revenue_of_nothing_is_empty():
    empty = spark.createDataFrame([], "order_id int, status string, order_date date, amount decimal(12,2)")
    assert daily_revenue(empty).count() == 0

for test in (test_daily_revenue_ignores_placed_orders, test_daily_revenue_of_nothing_is_empty):
    test()
    print("passed:", test.__name__)

**What to notice:** The first test checks values and the output schema, including `revenue` as `decimal(22,2)`: a change to `double`
would fail it. The second makes sure an empty day produces an empty result instead of an error. In a project these
functions live in `tests/` and pytest discovers them; the assertions are the test itself.

## Log what the job did

Use Python's `logging` on the driver for job-level messages: parameters, row counts at each boundary, rejected
rows, output paths, durations. Write the same numbers to a small run-metrics table, so you can chart them over
time and alert when today's count is far from normal. Spark's own logs (log4j) go to the driver and executor logs
of the cluster; set their level with `spark.sparkContext.setLogLevel`.

**Where it matters**
- **On-call:** "the job succeeded but wrote 0 rows" is visible in the metrics, not only in a complaint.
- **Capacity planning:** input size and duration per run.

**Watch out for**
- `logging` calls inside UDFs run on executors; their output is in executor logs, not on the driver.
- Each `count()` for logging is an extra job; count cached frames or reuse counts you already need.

### Example

Log row counts for a run and append them to a metrics table.

In [ ]:
log = logging.getLogger("orders_pipeline")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)

started = time.time()
source = standardize(orders).cache()
result = daily_revenue(source)
metrics = {"input_rows": source.count(), "output_rows": result.count()}
log.info("business_date=all input_rows=%d output_rows=%d", metrics["input_rows"], metrics["output_rows"])

metrics_path = os.path.join(WORK_DIR, "ops_run_metrics")
spark.createDataFrame(
    [("daily_revenue", "2026-01-15", metrics["input_rows"], metrics["output_rows"], round(time.time() - started, 2))],
    "job string, run_date string, input_rows long, output_rows long, seconds double",
).write.mode("append").parquet(metrics_path)
spark.read.parquet(metrics_path).show()
source = source.unpersist()

**What to notice:** The log line records the counts with the parameters that produced them, and the metrics table keeps one row per
run. An alert on `output_rows` far below its usual level catches a broken source on the night it breaks, not at
the next review.

## Check schemas at the boundaries

A job should fail early and clearly when its input breaks the contract: a missing column, a changed type, a
column that was renamed. Compare the incoming schema with the expected one before transforming, and list every
difference in the error. Extra columns can be allowed (additive changes); missing or changed ones should stop the
run.

**Where it matters**
- **Upstream changes** made without notice.
- **Before writing:** confirm the output matches the published table's schema.

**Watch out for**
- Type changes hidden by a cast (`string` read as `int` returns null) only show up as missing values later.
- Compare names case-sensitively if downstream systems do.

### Example

Check two deliveries against the expected schema: one adds a column, one renames and retypes columns.

In [ ]:
EXPECTED = {"order_id": "int", "status": "string", "order_ts": "timestamp", "amount": "decimal(12,2)"}


def check_schema(df, expected=EXPECTED, allow_extra=True):
    actual = dict(df.dtypes)
    problems = [f"missing column {name}" for name in expected if name not in actual]
    problems += [f"{name}: expected {kind}, got {actual[name]}" for name, kind in expected.items()
                 if name in actual and actual[name] != kind]
    if not allow_extra:
        problems += [f"unexpected column {name}" for name in actual if name not in expected]
    if problems:
        raise ValueError("schema check failed: " + "; ".join(problems))
    return True

good = orders.withColumn("currency", F.lit("EUR"))
bad = orders.withColumnRenamed("amount", "total").withColumn("order_id", F.col("order_id").cast("string"))
print("delivery with an extra column:", check_schema(good))
try:
    check_schema(bad)
except ValueError as error:
    print(error)

**What to notice:** The extra `currency` column is accepted as an additive change. The second delivery fails before any work is
done, and the message lists both problems at once: the missing `amount` and the `order_id` type change. That is
what an on-call engineer needs at 3 a.m.

## Package dependencies and submit the job

`spark-submit` starts the driver and asks the cluster manager for executors. Your code and its Python dependencies
must reach every executor:

- your package as a wheel or zip with `--py-files`;
- third-party libraries as a packed virtual environment (`venv-pack` or `conda-pack`) with `--archives`;
- JVM libraries (Kafka, Delta, JDBC drivers) with `--packages` or `--jars`.

```bash
spark-submit \
  --master yarn --deploy-mode cluster \
  --num-executors 20 --executor-cores 4 --executor-memory 16g \
  --conf spark.sql.shuffle.partitions=400 \
  --conf spark.pyspark.python=./environment/bin/python \
  --archives pyspark_env.tar.gz#environment \
  --py-files orders_pipeline-1.4.0-py3-none-any.whl \
  --packages io.delta:delta-spark_2.13:4.0.0 \
  jobs/daily_orders.py --business-date 2026-10-04 --env prod
```

On Databricks, a job task with a wheel and cluster libraries does the same; on Kubernetes, the dependencies are
usually baked into the container image.

**Where it matters**
- **Reproducible deployments:** the same wheel and environment in staging and production.
- **Libraries the cluster does not have,** such as a vendor SDK or a newer pandas.

**Watch out for**
- A package missing on executors fails only when a UDF first runs, with `ModuleNotFoundError` in a task.
- Mixing Python versions between driver and executors breaks pickled functions.

### Example

Read the command above from the bottom up: the script and its arguments, then where the JVM packages, the Python
package and the Python environment come from, then shuffle settings, resources, and finally where the driver runs.

## Choose a deploy mode and a cluster manager

In **cluster** deploy mode the driver runs inside the cluster, next to the executors; the machine that called
`spark-submit` can disconnect. In **client** mode the driver runs on the submitting machine, which suits
notebooks and interactive debugging. The **cluster manager** starts the processes:

| manager | typical setting |
| --- | --- |
| YARN | Hadoop clusters, Amazon EMR |
| Kubernetes | containerized platforms; executors are pods |
| standalone | Spark's own simple manager |
| Databricks, Dataproc, EMR Serverless | managed platforms that hide most of this |

**Dynamic allocation** (`spark.dynamicAllocation.enabled`) adds and removes executors with the workload, which
suits shared clusters and jobs with uneven stages.

**Where it matters**
- **Scheduled production jobs:** cluster mode, so the job does not depend on the scheduler host.
- **Shared clusters:** dynamic allocation returns idle executors to other teams.

**Watch out for**
- In client mode, a large `collect()` lands on the submitting machine, often a small scheduler host.
- Dynamic allocation needs shuffle data to outlive executors: an external shuffle service or shuffle tracking.

### Example

A nightly job submitted from Airflow should run in cluster mode: if the Airflow worker restarts mid-job, the driver
keeps running in the cluster. A notebook on the same cluster runs in client mode, so results come back to you.

## Orchestrate and retry safely

A scheduler (Airflow, Dagster, Databricks Workflows) runs the job with parameters, retries it on failure and runs
downstream jobs after it. Retries are only safe if the job is idempotent ([lesson 10](pyspark-10-batch-pipelines.ipynb)): a rerun for the
same date must overwrite or merge, never append twice. Keep each task small enough to rerun cheaply, and pass the
business date down the chain instead of letting each job compute it.

```python
# Airflow, simplified
SparkSubmitOperator(
    task_id="daily_orders",
    application="jobs/daily_orders.py",
    application_args=["--business-date", "{{ ds }}", "--env", "prod"],
    retries=2,
)
```

**Where it matters**
- **Daily batch pipelines** with dependencies between bronze, silver and gold jobs.
- **Backfills:** the scheduler reruns past dates with the same task.

**Watch out for**
- Retries on a non-idempotent job turn one failure into duplicated data.
- Alert on missing runs and on suspicious output counts, not only on failures.

### Example

`{{ ds }}` is the scheduled date, passed to the job as `--business-date`. A retry or a backfill for 4 October runs
exactly the same command, so it rewrites exactly the same partition.

## Choose a pipeline design for the workload

Common designs and the guarantee each must provide:

| requirement | design | key guarantee |
| --- | --- | --- |
| a full snapshot per day | batch job overwriting the business date | the date is replaced atomically, after checks |
| changes from an operational database | CDC into bronze, merge into silver | one row per key, newest version wins |
| events within minutes | Structured Streaming with `foreachBatch` merges | checkpoint plus idempotent sink |
| features for machine learning | one feature function for training and scoring | identical definitions, saved statistics |
| messy logs | permissive reads, quarantine, `VARIANT` in bronze | nothing lost, bad rows counted |

**Where it matters**
- **System design interviews** ask you to pick one of these and defend it.
- **New pipelines:** the requirement decides the design, not the tool you know best.

**Watch out for**
- Streaming adds operational cost; a scheduled `availableNow` stream or batch job is often enough.
- A z-score computed per batch at scoring time differs from the training one; save the training statistics.

### Example

Feature engineering in miniature: compute customer spend features, then scale them with statistics computed once
over the whole column and broadcast back, the values you would save for scoring.

In [ ]:
spend = orders.groupBy("customer_id").agg(F.count("*").alias("orders"), F.sum("amount").cast("double").alias("spend"))
stats = spend.agg(F.mean("spend").alias("mu"), F.stddev("spend").alias("sigma"))
features = (
    spend.crossJoin(F.broadcast(stats))
    .withColumn("spend_z", F.round((F.col("spend") - F.col("mu")) / F.col("sigma"), 3))
    .drop("mu", "sigma")
)
features.orderBy("customer_id").show()
print("saved for scoring:", stats.first().asDict())

**What to notice:** Customer 4, with the 1,180.00 bulk order, stands out with the highest z-score. The mean and standard deviation
were computed once over all customers; saving them and reusing them at scoring time keeps training and scoring
consistent. A pandas UDF computing `s.mean()` would have used one batch at a time instead.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why keep transformations free of reads and writes?

**A.** So they can be tested with small in-memory DataFrames and reused in notebooks, tests and jobs.

**Q2.** Why must the business date be a parameter?

**A.** So reruns and backfills write the same partition; deriving it from the clock makes output depend on when the job ran.

**Q3.** How do you share one SparkSession across tests?

**A.** A session-scoped pytest fixture in `conftest.py` that builds a local session once and stops it at the end.

**Q4.** How do Python dependencies reach the executors?

**A.** Your package with `--py-files`, third-party libraries in a packed environment with `--archives` (or a container image), JVM libraries with `--packages`.

**Q5.** Cluster or client deploy mode for a scheduled job?

**A.** Cluster mode: the driver runs inside the cluster and does not depend on the scheduler host.

**Q6.** What makes scheduler retries safe?

**A.** An idempotent job: a rerun for the same parameters overwrites or merges its own output instead of appending.

## Key takeaways

- Separate logic, I/O and entry point; test the logic with small DataFrames.
- Pass dates, environments and paths as parameters; read secrets from a secret store.
- Log counts and write run metrics; alert on unusual output, not just on failures.
- Check input schemas at the boundary and fail with every problem listed.
- Ship code and dependencies to executors explicitly; run scheduled jobs in cluster mode.
- Retries are safe only for idempotent jobs.

**Next:** Continue with [PySpark 15: interview concepts](pyspark-15-interview-concepts.ipynb).